In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import eos #EOS v1.0.16

from btosll_discontinuities import *

In [ ]:
@np.vectorize
def disc_EOS(s: float, case: str) -> np.cdouble:

    s_rescaled = s * 4.18**2
    eps = 1e-8

    p = eos.Parameters.Defaults()
    p.set("mass::c", 4.18 * np.sqrt(0.1))
    p.set("sb::mu", 4.18)

    kinematics_plus = {"Re{q2}": s_rescaled, "Im{q2}": eps}
    kinematics_minus = {"Re{q2}": s_rescaled, "Im{q2}": -eps}

    real_part = eos.Observable.make(f"b->s::Re{{F27}}(Re{{q2}},Im{{q2}})", p, eos.Kinematics(kinematics_plus), eos.Options({"contribution": case})).evaluate() - eos.Observable.make(f"b->s::Re{{F27}}(Re{{q2}},Im{{q2}})", p, eos.Kinematics(kinematics_minus), eos.Options({"contribution": case})).evaluate()
    imag_part = eos.Observable.make(f"b->s::Im{{F27}}(Re{{q2}},Im{{q2}})", p, eos.Kinematics(kinematics_plus), eos.Options({"contribution": case})).evaluate() - eos.Observable.make(f"b->s::Im{{F27}}(Re{{q2}},Im{{q2}})", p, eos.Kinematics(kinematics_minus), eos.Options({"contribution": case})).evaluate()


    return real_part + 1j * imag_part

In [ ]:
def plot_discontinuities(case: str, xmin: float, xmax: float, ymin: float, ymax: float, plotpoints: int = 300, savedata: bool = False, savefig: bool = False) -> None:
    c1 = (87/255,144/255,252/255)
    c2 = (248/255,156/255,32/255)
    c3 = (228/255,37/255,54/255)
    c4 = (150/255,74/255,139/255)

    if case == 'a':
        disc = f27a_disc
        plot_re = True
        sthr = 0
        sthr_label = r"0"
        sanom = 0.6
        sanom_label = r"0.6\,m_b^2"
    elif case == 'b':
        disc = f27b_disc
        plot_re = False
        sthr = 4
        sthr_label = r"4\,m_b^2"
        sanom = None
    elif case == 'c':
        disc = f27c_disc
        plot_re = True
        sthr = 0.4
        sthr_label = r"4\,m_c^2"
        sanom = 1
        sanom_label = r"m_b^2"
    elif case == 'd':
        disc = f27d_disc
        plot_re = False
        sthr = 0.4
        sthr_label = r"4\,m_c^2"
        sanom = None
    else:
        print("invalid case!")
        return None

    xarr = np.linspace(xmin+1e-4,xmax,plotpoints)
    yarr2loop = disc_EOS(xarr,case)
    yarrfit = disc(xarr)

    if savedata:
        if plot_re:
            np.savetxt(r"data/f27%s_disc.csv"%case,np.transpose([xarr,np.real(yarr2loop),np.imag(yarr2loop),np.real(yarrfit),np.imag(yarrfit)]),fmt="%.5f",header="s [m_b^2],Re disc 2-loop,Im disc 2-loop,Re disc fit,Im disc fit",delimiter=',')
        else:
            np.savetxt(r"data/f27%s_disc.csv"%case,np.transpose([xarr,np.imag(yarr2loop),np.imag(yarrfit)]),fmt="%.5f",header=r"s [m_b^2],Im disc 2-loop,Im disc fit",delimiter=',')
        
    if case == 'c':
        yarrfit2 = yarrfit[xarr>1]
        yarr2loop2 = yarr2loop[xarr>1]
        xarr2 = xarr[xarr>1]
        yarr2loop = yarr2loop[xarr<1]
        yarrfit = yarrfit[xarr<1]
        xarr = xarr[xarr<1]

    if plot_re:
        plt.plot(xarr,np.real(yarr2loop),label=r"Re disc 2-loop",c=c1,alpha=0.6,lw=5)
        plt.plot(xarr,np.real(yarrfit),label=r"Re disc fit",c=c4,lw=1,ls="dashed")
    plt.plot(xarr,np.imag(yarr2loop),label=r"Im disc 2-loop",c=c2,alpha=0.6,lw=5)
    plt.plot(xarr,np.imag(yarrfit),label=r"Im disc fit",c=c3,lw=1,ls="dashed")
    if case == 'c':
        plt.plot(xarr2,np.real(yarr2loop2),c=c1,alpha=0.6,lw=5)
        plt.plot(xarr2,np.real(yarrfit2),c=c4,lw=1,ls="dashed")
        plt.plot(xarr2,np.imag(yarr2loop2),c=c2,alpha=0.6,lw=5)
        plt.plot(xarr2,np.imag(yarrfit2),c=c3,lw=1,ls="dashed")
    plt.vlines(sthr,ymin,ymax,linestyles="dashed",colors="black",label=r"$s=%s$"%sthr_label)
    if sanom:
        plt.vlines(sanom,ymin,ymax,linestyles="dotted",colors="black",label=r"$s=%s$"%sanom_label)
    plt.xlim(xmin,xmax)
    plt.ylim(ymin,ymax)
    plt.xlabel(r"$s$ [$m_b^2$]")
    plt.ylabel(r"disc $F_{2(%s)}^{(7)}(s)$"%case)
    if case == 'b':
        plt.legend(loc="upper right")
    else:
        plt.legend(loc="lower right")
    if savefig:
        plt.savefig(r"figures/F27%s_disc.pdf"%case,format="pdf",bbox_inches='tight')
        plt.rcParams['savefig.dpi'] = 300
        plt.savefig(r"figures/F27%s_disc.png"%case,format="png",bbox_inches='tight')
    plt.show()

    return None

In [ ]:
plot_discontinuities("a",-0.1,1.25,-0.45,0.4,savefig=True,savedata=True)

In [ ]:
plot_discontinuities("b",3.7,10,-2.3,0.2,savefig=True,savedata=True)

In [ ]:
plot_discontinuities("c",0,2,-80,60,savefig=True,savedata=True)

In [ ]:
plot_discontinuities("d",0,2,-1,4,savefig=True,savedata=True)